In [26]:
import pandas as pd
import numpy as np
import mlflow
import joblib
from pathlib import Path
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

PROCESSED = Path("../data/processed")
MODELS = Path("../models")

df = pd.read_parquet(PROCESSED / "features.parquet")
print(df.shape)

(2905, 10)


In [27]:
FEATURES = [
    "home_form", "away_form",
    "home_fixture_density", "away_fixture_density",
    "elo_delta", "home_advantage",
    "B365H", "B365D", "B365A"
]

le = LabelEncoder()
df["result_enc"] = le.fit_transform(df["result"])
X = df[FEATURES]
y = df["result_enc"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

Train: 2324, Test: 581


In [28]:
results = []
for n in [50, 100, 200, 300]:
    model = XGBClassifier(n_estimators=n, max_depth=4,
                          learning_rate=0.05, eval_metric="mlogloss",
                          random_state=42)
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    results.append({"n_estimators": n, "accuracy": acc})
    print(f"n_estimators={n} → accuracy={acc:.4f}")

pd.DataFrame(results)

n_estimators=50 → accuracy=0.5077
n_estimators=100 → accuracy=0.5043
n_estimators=200 → accuracy=0.5077
n_estimators=300 → accuracy=0.4957


,n_estimators,accuracy
0,50,0.507745
1,100,0.504303
2,200,0.507745
3,300,0.495697


In [29]:
results2 = []
for depth in [3, 4, 5, 6]:
    model = XGBClassifier(n_estimators=200, max_depth=depth,
                          learning_rate=0.05, eval_metric="mlogloss",
                          random_state=42)
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    results2.append({"max_depth": depth, "accuracy": acc})
    print(f"max_depth={depth} → accuracy={acc:.4f}")

pd.DataFrame(results2)

max_depth=3 → accuracy=0.5129
max_depth=4 → accuracy=0.5077
max_depth=5 → accuracy=0.5129
max_depth=6 → accuracy=0.4854


,max_depth,accuracy
0,3,0.512909
1,4,0.507745
2,5,0.512909
3,6,0.485370


In [30]:
print("Best params from experiments:")
print("  n_estimators: check MLflow UI")
print("  max_depth: check MLflow UI")
print("\nRun: mlflow ui --backend-store-uri ../mlruns")

Best params from experiments:
  n_estimators: check MLflow UI
  max_depth: check MLflow UI

Run: mlflow ui --backend-store-uri ../mlruns
